<a href="https://colab.research.google.com/github/durvanshi/Gen-AI-Practicals/blob/main/genAi_prac4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip -q install tensorflow pandas numpy

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow Version:", tf.__version__)


corpus = """
deep learning is powerful
deep learning is useful
deep learning is widely used
deep learning is a branch of machine learning
machine learning is useful
machine learning is powerful
machine learning is widely used
artificial intelligence is changing the world
artificial intelligence is useful
artificial intelligence is powerful
data science is an important field
data science is widely used
data science is useful
python is easy to learn
python is widely used
python is powerful
python programming is easy
python programming is useful
lstm is a neural network
lstm is useful for sequence prediction
lstm is powerful for text prediction
lstm is widely used in natural language processing
natural language processing is useful
natural language processing is powerful
natural language processing is widely used
the model predicts the next word
the model learns from text
the model is trained using data
the system predicts useful words
the system predicts the next word
the keyboard suggests the next word
the keyboard is useful
the keyboard is smart
students learn machine learning
students learn deep learning
students use python programming
students study artificial intelligence
i love machine learning
i love deep learning
i like python programming
i like data science
we use artificial intelligence
we use machine learning
we use deep learning
technology is changing the world
technology is useful
technology is powerful
technology is widely used
"""


# 2. TEXT PREPROCESSING AND TOKENIZATION

tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1

print("Vocabulary Size:", total_words)

# Convert sentences into numerical sequences
input_sequences = []

for line in corpus.strip().split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)


# 3. PADDING

max_sequence_len = max(len(x) for x in input_sequences)

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding='pre'
    )
)

# Input and output
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# Convert output to one-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Maximum Sequence Length:", max_sequence_len)
print("Training Samples:", len(X))


# 4. LSTM MODEL

model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=64,
        input_length=max_sequence_len - 1
    ),

    LSTM(128),

    Dense(total_words, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()



# 5. TRAIN THE MODEL


print("\nTraining the LSTM model...\n")

history = model.fit(
    X,
    y,
    epochs=100,
    batch_size=16,
    verbose=0
)

print("Training Completed!")
print("Final Accuracy:",
      round(history.history['accuracy'][-1] * 100, 2), "%")


# 6. NEXT WORD PREDICTION FUNCTION


def predict_next_words(seed_text, top_k=3):

    # Convert user sentence to tokens
    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    # Check whether words are present in vocabulary
    if len(token_list) == 0:
        return []

    # Padding
    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding='pre'
    )

    # Predict probabilities
    probabilities = model.predict(token_list, verbose=0)[0]

    # Get top K word indexes
    top_indices = np.argsort(probabilities)[-top_k:][::-1]

    predictions = []

    for index in top_indices:

        word = None

        for w, idx in tokenizer.word_index.items():
            if idx == index:
                word = w
                break

        if word is not None:
            predictions.append(
                (word, float(probabilities[index]))
            )

    return predictions


# 7. TAKE USER INPUT


print("\n==============================================")
print("       NEXT WORD PREDICTION SYSTEM")
print("==============================================")

user_sentence = input(
    "\nEnter a sentence (example: Deep learning is): "
)

top_k = 3

predictions = predict_next_words(
    user_sentence,
    top_k
)


# 8. DISPLAY TOP-K PREDICTIONS


if len(predictions) == 0:

    print("\nSorry! The entered words are not present")
    print("in the training vocabulary.")

else:

    print("\nTop-K Predicted Words:\n")

    result = []

    for word, probability in predictions:

        result.append({
            "Sentence": user_sentence,
            "Predicted Word": word,
            "Probability": f"{probability * 100:.2f}%"
        })

    df = pd.DataFrame(result)

    display(df)

# 9. SINGLE BEST PREDICTION


if len(predictions) > 0:

    best_word = predictions[0][0]

    print("\nPredicted Next Word:", best_word)

    print(
        "\nCompleted Sentence:",
        user_sentence + " " + best_word
    )

TensorFlow Version: 2.21.0
Vocabulary Size: 59
Maximum Sequence Length: 8
Training Samples: 177


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training the LSTM model...

Training Completed!
Final Accuracy: 75.71 %

       NEXT WORD PREDICTION SYSTEM

Enter a sentence (example: Deep learning is): deep learning is

Top-K Predicted Words:



,Sentence,Predicted Word,Probability
0,deep learning is,widely,34.05%
1,deep learning is,powerful,26.13%
2,deep learning is,useful,23.58%



Predicted Next Word: widely

Completed Sentence: deep learning is widely
